# 最高到達accuracyを取得する

In [1]:
from datetime import date

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json
import re
import sys
from collections import defaultdict

project_root = os.path.join(os.getcwd(), "..")
result_dir = os.path.join(project_root, "results")


def load_file_and_extract_max_accs(date, lr, target_concept_config_name, model_size, result_dir):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""
    # if target_concept_config_type != '':
    #     target_concept_config_type = '-' + target_concept_config_type
    # dirname_pattern1 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_target_concepts_initvecwith(.*)"
    # dirname_pattern2 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-seed(\\d+)_target_concepts_initvecwith(.*)"

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    initVecType_to_seed_to_maxacc = defaultdict(dict)
    initVecType_to_seed_to_epoch_at_maxacc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                # print(dirname)
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        # print(f"Processing directory: {dirname}, initVecType: {initVecType}, seed: {seed}")
        
        if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
            # print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
            continue
        with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
            concept_to_epoch_to_scores = json.load(f)

        # 各conceptについて、回したepochs中の最高到達accuracyとそのepochを抽出
        max_accs = []
        epochs_at_maxacc = []
        for epoch_to_scores in concept_to_epoch_to_scores.values():
            max_accuracy = 0
            max_epoch_at_maxacc = 0
            for epoch, scores in epoch_to_scores.items():
                # print(f"Epoch: {epoch}, Accuracy: {scores['accuracy']}")
                if scores['accuracy'] > max_accuracy:
                    max_accuracy = scores['accuracy']
                    max_epoch_at_maxacc = epoch
            max_accs.append(max_accuracy)
            epochs_at_maxacc.append(max_epoch_at_maxacc)
        # print(f"Max Accuracies: {max_accs}")
        # print(f"Epochs at Max Accuracies: {epochs_at_maxacc}")
        initVecType_to_seed_to_maxacc[initVecType][seed] = max_accs
        initVecType_to_seed_to_epoch_at_maxacc[initVecType][seed] = epochs_at_maxacc
    return initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc


## 12B

In [2]:

date = '20260419'
model_size = '12'
lr = 0.003

target_concept_config_name = 'target_concepts_mini_13.json'


In [74]:

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

seeds = set([seed for seed_to_maxacc in initVecType_to_seed_to_maxacc.values() for seed in seed_to_maxacc.keys()])
initVecType_lst = sorted(list(initVecType_to_seed_to_maxacc.keys()))

for seed in seeds:
    print(f"Seed: {seed}")
    for initVecType in initVecType_lst:
        max_accs = initVecType_to_seed_to_maxacc[initVecType].get(seed, None)
        if max_accs:
            mean_max_acc = np.mean(max_accs)
            std_max_acc = np.std(max_accs)
            print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")

print('\n' + '-' * 100 + '\n')

for seed in seeds:
    print(f"Seed: {seed}")
    for initVecType in initVecType_lst:
        epochs_at_maxacc = initVecType_to_seed_to_epoch_at_maxacc[initVecType].get(seed, None)

        if epochs_at_maxacc:
            mean_epoch_at_maxacc = np.mean([int(e) for e in epochs_at_maxacc])
            std_epoch_at_maxacc = np.std([int(e) for e in epochs_at_maxacc])
        print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Epoch number at Max Accuracy: {mean_epoch_at_maxacc:.2f}, Std: {std_epoch_at_maxacc:.2f}")


Seed: 0
	InitVecType: CatCent_by_WikiSummRepeatHSMix_noRand_layer12, Seed: 0, Mean Max Accuracy: 0.7210, Std: 0.1694
	InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7141, Std: 0.1251
	InitVecType: otherCatCent_by_WikiSummRepeatHSMix_noRand_layer12, Seed: 0, Mean Max Accuracy: 0.6619, Std: 0.1402
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7207, Std: 0.1748
Seed: 1
	InitVecType: CatCent_by_WikiSummRepeatHSMix_noRand_layer12, Seed: 1, Mean Max Accuracy: 0.7065, Std: 0.1392
	InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7410, Std: 0.1401
	InitVecType: otherCatCent_by_WikiSummRepeatHSMix_noRand_layer12, Seed: 1, Mean Max Accuracy: 0.7189, Std: 0.1545
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.6952, Std: 0.1255
Seed: 2
	InitVecType: CatCent_by_WikiSummRepeatHSMix_noRand_layer12, Seed: 2, Mean Max Accuracy: 0.7

In [80]:

acc_cat_norand = []
acc_other_norand = []

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

seeds = set([seed for seed_to_maxacc in initVecType_to_seed_to_maxacc.values() for seed in seed_to_maxacc.keys()])
initVecType_lst = list(initVecType_to_seed_to_maxacc.keys())

for seed in seeds:
    print(f"Seed: {seed}")
    # for initVecType in initVecType_lst:

    max_accs = initVecType_to_seed_to_maxacc["CatCent_by_WikiSummRepeatHSMix_noRand_layer12"].get(seed, None)
    if max_accs:
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
        acc_cat_norand.append(mean_max_acc)
    max_accs = initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummRepeatHSMix_noRand_layer12"].get(seed, None)
    if max_accs:
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
        acc_other_norand.append(mean_max_acc)


Seed: 0
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7210, Std: 0.1694
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.6619, Std: 0.1402
Seed: 1
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7065, Std: 0.1392
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7189, Std: 0.1545
Seed: 2
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.7448, Std: 0.1388
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.6957, Std: 0.1728
Seed: 3
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 3, Mean Max Accuracy: 0.7134, Std: 0.1288
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 3, Mean Max Accuracy: 0.7531, Std: 0.1426
Seed: 4
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, 

In [32]:
import numpy as np
from scipy import stats


acc_cat_norand = np.array(acc_cat_norand)
acc_other_norand = np.array(acc_other_norand)

diff = acc_cat_norand - acc_other_norand

print("diff:", [float(round(x, 4)) for x in diff])
print(f"mean diff: {np.mean(diff):.4f}")
print(f"std diff : {np.std(diff):.4f}")

# paired t-test
t_stat, p_value = stats.ttest_1samp(diff, 0)

print(f"t = {t_stat:.4f}")
print(f"p = {p_value:.4f}")

diff: [0.059, -0.0124, 0.0491, -0.0397, 0.0015, -0.0215, 0.0327, 0.0047, 0.0137, -0.0872]
mean diff: -0.0000
std diff : 0.0412
t = -0.0008
p = 0.9994


In [81]:

acc_cat = []
acc_other = []

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

seeds = set([seed for seed_to_maxacc in initVecType_to_seed_to_maxacc.values() for seed in seed_to_maxacc.keys()])
initVecType_lst = list(initVecType_to_seed_to_maxacc.keys())

for seed in seeds:
    print(f"Seed: {seed}")
    # for initVecType in initVecType_lst:

    max_accs = initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"].get(seed, None)
    if max_accs:
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
        acc_cat.append(mean_max_acc)
    max_accs = initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"].get(seed, None)
    if max_accs:
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"\tInitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
        acc_other.append(mean_max_acc)


Seed: 0
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7141, Std: 0.1251
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Mean Max Accuracy: 0.7207, Std: 0.1748
Seed: 1
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.7410, Std: 0.1401
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Mean Max Accuracy: 0.6952, Std: 0.1255
Seed: 2
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.7191, Std: 0.1147
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Mean Max Accuracy: 0.7093, Std: 0.1624
Seed: 3
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 3, Mean Max Accuracy: 0.7172, Std: 0.1353
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 3, Mean Max Accuracy: 0.7437, Std: 0.1420
Seed: 4
	InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12, 

In [82]:
import numpy as np
from scipy import stats


acc_cat = np.array(acc_cat)
acc_other = np.array(acc_other)

diff = acc_cat - acc_other

print("diff:", [float(round(x, 4)) for x in diff])
print(f"mean diff: {np.mean(diff):.4f}")
print(f"std diff : {np.std(diff):.4f}")

# paired t-test
t_stat, p_value = stats.ttest_1samp(diff, 0)

print(f"t = {t_stat:.4f}")
print(f"p = {p_value:.4f}")

diff: [-0.0066, 0.0458, 0.0098, -0.0265, -0.0312, 0.0299, 0.0201, 0.0093, 0.0097, 0.0327, 0.0013, 0.0357, -0.0372, -0.0485, -0.0103, -0.0227, 0.011, 0.0127, -0.0028, -0.0164]
mean diff: 0.0008
std diff : 0.0251
t = 0.1372
p = 0.8923


## epoch4固定でaccuracyを計算する

In [3]:
sys.path.append(project_root)

from utils.gemma_train_and_test_utils import calculate_metrics

def get_concept_to_epoch_score(concept_result_dir, epoch):
    concept_to_results = {}
    for dirname in os.listdir(concept_result_dir):
        if dirname.endswith(".json"):
            continue
        concept = dirname

        # このconceptに対するepochの結果を読み込む
        logit_score_path = os.path.join(concept_result_dir, dirname, f"logit-scored_epoch{epoch}.json")
        
        if not os.path.exists(logit_score_path):
            # raise FileNotFoundError(f"Logit score file not found: {logit_score_path}")
            # print(f"Logit score file not found: {logit_score_path}. Skipping this concept.")
            continue
            
        with open(logit_score_path, 'r') as f:
            results = json.load(f)
        y_true_lst = [res['label'] for res in results]
        y_pred_lst = [res['pred'] for res in results]

        # score計算
        metrics = calculate_metrics(y_pred_lst, y_true_lst)

        concept_to_results[concept] = metrics

        # print(f"  Epoch {epoch}: Accuracy: {metrics['accuracy']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['F1']:.4f}")
    
    return concept_to_results





def load_file_and_calc_epoch_accs(date, epoch, lr, target_concept_config_name, model_size, result_dir, print_flag=False):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"

    initvectype_to_seed_to_acc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        concept_to_results = get_concept_to_epoch_score(os.path.join(result_dir, dirname), epoch)
        if len(concept_to_results) == 0:
            # print(f"No valid concepts with logit score files found for directory: {dirname}. Skipping.")
            continue
        sum_acc = 0
        for concept, score in concept_to_results.items():
            acc = score['accuracy']
            sum_acc += acc

        avg_acc = sum_acc / len(concept_to_results)
        initvectype_to_seed_to_acc[initVecType][seed] = avg_acc
        if print_flag:
            print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {avg_acc:.4f}")

    return initvectype_to_seed_to_acc


In [85]:
# epoch = 2

date = '20260419'

for epoch in range(1, 5):

    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)



    # for seed in range(10):
    # ** Cat **
    initVecType = 'CatCent_by_WikiSummaryRepeatHSMixed_layer12'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(20): 
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)
        
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    cat_acc_lst = acc_lst


    # ** otherCat **
    initVecType = 'otherCatCent_by_WikiSummaryRepeatHSMixed_layer12'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(20):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)
    
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    othercat_acc_lst = acc_lst


    diff_lst = [cat_acc - othercat_acc for cat_acc, othercat_acc in zip(cat_acc_lst, othercat_acc_lst)]
    print(f"plus diff: {len([d for d in diff_lst if d > 0])} / {len(diff_lst)}")
    print(f"minus diff: {len([d for d in diff_lst if d < 0])} / {len(diff_lst)}")
    print(diff_lst)
    print('\n==============================\n')
    

InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.5914062632534601, 0.6699913077915853, 0.6349333455406955, 0.64715784774515, 0.6223131051716646, 0.6222908583909793, 0.6066667703019814, 0.6203023305785175, 0.6172711505096014, 0.6457207751719418, 0.6069027339079626, 0.6548376632129329, 0.6123948511040122, 0.6292581389703767, 0.6360458732243046, 0.6486089418928077, 0.6191270220713576, 0.6367355677324961, 0.6079196017143279, 0.6010606732744863]
accuracy at epoch 1: 0.6265 ± 0.0196

InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.6379769517823364, 0.6288013271204212, 0.600077384950703, 0.6694235115528713, 0.6292458348265171, 0.5887292628519366, 0.5842519255267296, 0.6157109900082276, 0.6159500990709845, 0.5715361389884356, 0.6125629941080825, 0.654213545675203, 0.6945802248115066, 0.6806941794216868, 0.652295126951558, 0.6883810155741051, 0.6274736282469742, 0.6104169618611399, 0.6380089271830942, 0.6373385799905992]
accuracy at epoch 1: 0.6319 ± 0.0332
plus diff

In [90]:
# epoch = 2

layer = 12
seed_num = 20

for epoch in range(1, 5):
    print(f"Epoch: {epoch}")


    date = '20260419'
    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)

    # ** Cat **
    initVecType = f'CatCent_by_WikiSummaryRepeatHSMixed_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num): 
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)
        
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    cat_acc_lst = acc_lst


    # ** otherCat **
    initVecType = f'otherCatCent_by_WikiSummaryRepeatHSMixed_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)
    
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    othercat_acc_lst = acc_lst


    # diff
    diff_lst = [cat_acc - othercat_acc for cat_acc, othercat_acc in zip(cat_acc_lst, othercat_acc_lst)]
    print(f"plus diff: {len([d for d in diff_lst if d > 0])} / {len(diff_lst)}")
    print(f"minus diff: {len([d for d in diff_lst if d < 0])} / {len(diff_lst)}")
    print(diff_lst)
    print('\n==============================\n')
    

    

    date = '20260418'
    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)
    # ** norm_rand_vocab **
    initVecType = f'norm_rand_vocab_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    norm_rand_vocab_acc_lst = acc_lst



    # diff
    diff_lst = [cat_acc - normrand_acc for cat_acc, normrand_acc in zip(cat_acc_lst, norm_rand_vocab_acc_lst)]
    print(f"plus diff: {len([d for d in diff_lst if d > 0])} / {len(diff_lst)}")
    print(f"minus diff: {len([d for d in diff_lst if d < 0])} / {len(diff_lst)}")
    print(diff_lst)
    print('\n==============================\n')



    # ** zero **
    initVecType = f'zero_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    zero_acc_lst = acc_lst



    # diff
    diff_lst = [cat_acc - zero_acc for cat_acc, zero_acc in zip(cat_acc_lst, zero_acc_lst)]
    print(f"plus diff: {len([d for d in diff_lst if d > 0])} / {len(diff_lst)}")
    print(f"minus diff: {len([d for d in diff_lst if d < 0])} / {len(diff_lst)}")
    print(diff_lst)
    print('\n==============================\n==============================\n')
    

Epoch: 1
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.5914062632534601, 0.6699913077915853, 0.6349333455406955, 0.64715784774515, 0.6223131051716646, 0.6222908583909793, 0.6066667703019814, 0.6203023305785175, 0.6172711505096014, 0.6457207751719418, 0.6069027339079626, 0.6548376632129329, 0.6123948511040122, 0.6292581389703767, 0.6360458732243046, 0.6486089418928077, 0.6191270220713576, 0.6367355677324961, 0.6079196017143279, 0.6010606732744863]
accuracy at epoch 1: 0.6265 ± 0.0196

InitVecType: otherCatCent_by_WikiSummaryRepeatHSMixed_layer12
[0.6379769517823364, 0.6288013271204212, 0.600077384950703, 0.6694235115528713, 0.6292458348265171, 0.5887292628519366, 0.5842519255267296, 0.6157109900082276, 0.6159500990709845, 0.5715361389884356, 0.6125629941080825, 0.654213545675203, 0.6945802248115066, 0.6806941794216868, 0.652295126951558, 0.6883810155741051, 0.6274736282469742, 0.6104169618611399, 0.6380089271830942, 0.6373385799905992]
accuracy at epoch 1: 0.6319 ± 0.0332


In [5]:
# epoch = 2

layer = 12
seed_num = 10

for epoch in range(1, 5):
    print(f"Epoch: {epoch}")

    date = '20260418'
    initvectype_to_seed_to_acc = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)
    # ** norm_rand_vocab **
    initVecType = f'norm_rand_vocab_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    norm_rand_vocab_acc_lst = acc_lst



    # ** zero **
    initVecType = f'zero_layer{layer}'
    print(f"InitVecType: {initVecType}")
    acc_lst = []
    for seed in range(seed_num):
        seed_to_acc = initvectype_to_seed_to_acc.get(initVecType, {})
        acc = seed_to_acc.get(seed, None)
        if acc is not None:
            acc_lst.append(acc)
            # print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
        else:
            print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
    print(acc_lst)

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    zero_acc_lst = acc_lst

    print('\n==============================\n==============================\n')
    

Epoch: 1
InitVecType: norm_rand_vocab_layer12
[0.5877783877011918, 0.6125625581591442, 0.6243171282314388, 0.5875983165489136, 0.6394275312337115, 0.5736421943034672, 0.643990115889439, 0.5441013108862109, 0.636321388313437, 0.5877425588623955]
accuracy at epoch 1: 0.6037 ± 0.0311
InitVecType: zero_layer12
[0.5807310320347437, 0.5247740687947468, 0.5500913405385442, 0.5873729160701612, 0.5177735848601164, 0.5488742213298244, 0.5277451926465028, 0.570430436880718, 0.5608318921098954, 0.5359881036031047]
accuracy at epoch 1: 0.5505 ± 0.0229


Epoch: 2
InitVecType: norm_rand_vocab_layer12
[0.6522056324870152, 0.6176438472469286, 0.6591828253561439, 0.6301544839483104, 0.6556122799857147, 0.6120959458699112, 0.6620245960055894, 0.6125848470979816, 0.6129906212304483, 0.6274452281321072]
accuracy at epoch 2: 0.6342 ± 0.0198
InitVecType: zero_layer12
[0.6065666112358944, 0.5601345132392684, 0.5608221209487481, 0.6037289219549682, 0.5607678639481555, 0.5811184516291097, 0.5521206651531387, 0.